# 📘 Notebook 13 · 链上 AI 与 ZKML

> Notebook 09-11 学了"用 AI 做量化"，Notebook 12 学了 Web3 最新生态。这一节把两条线**真正合起来**：让 AI 的计算可验证、可上链、可激励。

**学完你会：**

- 理解 ZK 证明的核心思想（不需要懂数学）
- 知道 ZKML 是什么、能干什么、有什么限制
- 知道链上 AI 的三种实现路径（ZKML / OPML / TEE）
- 知道 AI Agent 怎么在链上经济中运行
- 看清 AI × Crypto 的真实求职机会

**预计 5-7 小时（重在概念）**

**前置**：Notebook 09 + 12。

## 1. 零知识证明：5 分钟看懂

### 1.1 一句话定义

**让 Bob 相信"我知道某个秘密"，但不告诉 Bob 这个秘密本身。**

### 1.2 经典例子：阿里巴巴洞穴

一个 C 字形山洞，左右两条岔路在尽头被一扇魔法门隔开。Alice 知道密语能开门，想向 Bob 证明，但不愿透露密语。

```
       入口
      /    \
    左路   右路
     |       |
    [魔法门  -]
```

**协议**：
1. Bob 在洞外等，Alice 走进洞里随便选一条路
2. Bob 进入洞口，随机喊"从左边出来"或"从右边出来"
3. Alice 如果知道密语，永远能从指定方向出来；如果不知道，只有 50% 概率
4. 重复 20 次，骗过 Bob 的概率 < 1/百万

**核心思想**：通过**重复挑战**让作弊概率趋近于 0，但**永远不暴露密语**。

### 1.3 在密码学中的形式

**Prover**（证明者）想证明：
- "我知道一个 $x$，使得 $f(x) = y$"

**Verifier**（验证者）希望：
- **Completeness**：如果 Prover 真知道 $x$，Verifier 一定接受
- **Soundness**：如果 Prover 不知道 $x$，Verifier 几乎一定拒绝
- **Zero-Knowledge**：Verifier 看完证明，对 $x$ 一无所知

### 1.4 两大主流方案

| 方案 | 全称 | 特点 | 代表 |
|---|---|---|---|
| **zk-SNARK** | Succinct Non-interactive Argument of Knowledge | 证明小、验证快、需要 trusted setup | Groth16、PLONK、Halo2 |
| **zk-STARK** | Scalable Transparent ARgument of Knowledge | 不需要 trusted setup、抗量子、证明大 | StarkWare |

**对开发者：** 大部分时候用现成库（[circom](https://github.com/iden3/circom)、[halo2](https://github.com/zcash/halo2)、[Cairo](https://www.cairo-lang.org)），**不需要懂底层椭圆曲线 + Pairing**。

### 1.5 ZK 的"超能力"

- **L2**：把上千笔 L2 交易压成一个 ZK 证明扔到 L1，全部一次性验证
- **隐私交易**：Tornado Cash、Aztec、Railway 用 ZK 隐藏发送方/金额
- **链下计算 + 链上验证**：复杂计算放链下（便宜），链上只验证证明（贵但快）
- **ZKML**：训练/推理的"诚实性"可证明 ← **本节重点**

## 2. ZKML：为什么要把 ML 和 ZK 结合

### 2.1 问题：链上的 AI 都不可信

假设一个 DeFi 协议想用 AI 判断"这笔贷款风险高不高"：

```
方案 A: AI 模型跑在链上
  ⛔ Gas 费爆炸（一次推理 $10000+）

方案 B: AI 模型跑在链下，把结果发到链上
  ⛔ 你怎么知道结果是真的？运营方可能造假
```

→ **链上 AI 的核心问题：可验证性 + 经济性**。

### 2.2 ZKML 的方案

**让 AI 推理结果附带 ZK 证明**：

```
输入: x (公开)
模型权重: W (可公开或保密)
输出: y = Model(x; W)
   ↓
ZK 证明: "我用 W 跑 Model 得到 y，这个计算是诚实的"
   ↓
链上验证证明 → 接受 y
```

**好处**：
- 链下重计算（cheap）
- 链上轻量验证（cheap）
- 数学保证不可造假

### 2.3 ZKML 能做的真实场景

| 场景 | 例子 |
|---|---|
| **链上保险定价** | AI 算保费，证明算法没作弊 |
| **去中心化预测市场** | AI 给体育/选举打分，可证明 |
| **链上信用评分** | AI 评信用，但保护用户数据 |
| **可验证 NFT 生成** | 证明这个 NFT 是 AI 真按规则生成的 |
| **AI 比赛公平性** | Bittensor 等矿工奖励分配 |
| **机器人识别** | 证明"这个动作是 AI 做的"或"是人做的" |
| **保密推理** | 数据/模型保密，但结果可信 |

### 2.4 ZKML 不能做的

- **训练**：训练一个大模型需要的 ZK 电路太大，目前不可行
- **大模型推理**：GPT-4 级别模型的 ZK 证明现在要几小时甚至几天
- **超低延迟**：每次推理生成证明都比直接推理慢 100-1000 倍

**现实**：ZKML 现在只能用于**小型模型（MLP、小型 CNN、决策树）**，大模型还在等硬件 + 算法突破。

## 3. ZKML 主要项目

### 3.1 项目对比

| 项目 | 路线 | 进度 |
|---|---|---|
| **Modulus Labs** | 通用 ZKML 平台 | 已部署小模型应用（Upshot、Worldcoin） |
| **Giza** | StarkNet 上的 ZKML | 已上线 |
| **EZKL** | 把 ONNX 模型转 Halo2 电路 | 开源最活跃 |
| **Risc Zero** | zkVM，可证明任意计算 | 已上线 |
| **Ritual** | AI 协处理器（不是纯 ZK） | 主网 2024 |
| **Aizel Network** | TEE + ZK 混合 | 早期 |

### 3.2 EZKL 入门看一眼

> EZKL 是最容易上手的 ZKML 工具，让你把 PyTorch 模型转成 ZK 电路。

工作流：
```
1. PyTorch 训练模型
2. 导出 ONNX
3. EZKL 转成 Halo2 电路
4. 生成 setup（trusted setup 或 PLONK）
5. 给定输入，生成 proof
6. 验证 proof（链下或链上）
```

代码示例（教学伪代码）：

```python
import ezkl
import torch.onnx

# 1. 训练一个简单模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)
    def forward(self, x):
        return self.fc(x)

model = SimpleNet()
# ... train ...

# 2. 导出 ONNX
dummy = torch.randn(1, 10)
torch.onnx.export(model, dummy, "model.onnx")

# 3. EZKL 生成电路
ezkl.gen_settings("model.onnx", "settings.json")
ezkl.compile_circuit("model.onnx", "model.compiled", "settings.json")

# 4. setup
ezkl.setup("model.compiled", "vk.key", "pk.key")

# 5. 生成 proof
ezkl.prove(
    witness_path="witness.json",   # 包含输入和输出
    compiled_circuit_path="model.compiled",
    pk_path="pk.key",
    proof_path="proof.json",
)

# 6. 链上验证（Solidity 合约）
# EZKL 还可以生成 Solidity 验证合约，部署到链上
ezkl.create_evm_verifier("vk.key", "Verifier.sol")
```

**实际跑通需要**：
- 安装 `ezkl`（Rust 工具）
- GPU 加速（CPU 太慢）
- 模型不能太大（< 1M 参数最佳）

## 4. OPML：ZKML 的替代方案

### 4.1 OPML 思想

ZKML 慢、贵。**OPML（Optimistic ML）** 借用 Optimistic Rollup 思路：

```
1. 节点声称："我用 Model M 算出来 y"
2. 链上"乐观地"接受这个结果
3. 在挑战期内（如 7 天），任何人可以发起争议
4. 争议时：双方"分而治之"，找出第一步计算分歧的地方
5. 链上只验证那一步是否正确
```

→ **大部分时候不需要任何昂贵的 ZK 计算**，只在争议时才执行小段验证。

### 4.2 代表项目

**ORA（OPML 主要推动者）**：
- 用 OPML 在链上提供 LLM 推理服务
- 已上线 ORA Network，可在合约内调用 OpenAI / Llama / Stable Diffusion

**示例：**

```solidity
contract MyApp {
    IORA ora = IORA(0x...);

    function classify(string memory text) public {
        uint256 requestId = ora.requestCallback(
            "llama2-70b",
            text,
            address(this),
            "onAIResult(uint256,string)"
        );
    }

    function onAIResult(uint256 reqId, string memory result) external {
        // 这里就拿到了 AI 推理结果
        require(msg.sender == address(ora));
        // ... 处理 ...
    }
}
```

### 4.3 OPML vs ZKML 对比

| 维度 | ZKML | OPML |
|---|---|---|
| **延迟** | 慢（生成 proof 要 1s - 数小时）| 快（链上即时确认）|
| **成本** | 链下计算贵 | 链下计算正常成本 |
| **大模型支持** | 几乎不支持 | 支持任意大小 |
| **信任假设** | 数学证明，0 信任 | "至少一个诚实挑战者"假设 |
| **挑战期** | 无 | 通常 7 天 |

**结论**：现阶段大模型走 OPML，小模型走 ZKML。**头部团队都在两条路上同时投入**。

## 5. TEE：第三种路径

### 5.1 TEE 是什么

**Trusted Execution Environment**（可信执行环境）：CPU 提供的"隔离飞地"，里面跑的代码连主机操作系统都看不到。

代表：Intel SGX、AMD SEV、ARM TrustZone、Nvidia H100 confidential computing

### 5.2 用 TEE 做链上 AI

```
1. 模型权重 + 用户数据上传到 TEE
2. TEE 内执行推理（外部看不到）
3. TEE 用硬件密钥签名结果
4. 链上验证签名，确认推理是在真实 TEE 里跑的
```

**好处**：
- 速度快（接近原生 GPU/CPU）
- 支持大模型
- 数据隐私

**坏处**：
- 信任硬件厂商（Intel、AMD、Nvidia）
- SGX 历史上有过漏洞（Foreshadow、SGAxe）
- 中心化风险

### 5.3 代表项目

- **Phala Network**：TEE + 区块链
- **Marlin**：TEE 用于 DeFi 隐私
- **Oasis Network**：TEE 用于隐私 dApp
- **Aizel Network**：TEE + ZKP 混合

### 5.4 ZK / OP / TEE 三选一怎么选

```
小模型 + 高安全要求      → ZKML
大模型 + 可接受挑战期    → OPML
任何模型 + 信任硬件      → TEE
要数据隐私              → ZKML 或 TEE
要可审计性             → ZKML 或 OPML
要速度                  → TEE
```

**实际产品**：经常**混合用**——OPML + TEE，或 ZKML + 多签验证。

## 6. AI Agent 在 Web3 中的角色

### 6.1 AI Agent 经济的关键问题

一个真正自主的 AI Agent 需要解决：

| 问题 | 传统方案 | Web3 方案 |
|---|---|---|
| **怎么收钱** | 不行（没银行账户） | crypto 钱包 |
| **怎么付钱** | 不行 | 钱包 + 智能合约 |
| **怎么有身份** | 不行 | DID / ENS / SBT |
| **怎么获得奖励** | 公司发工资 | token 激励 |
| **怎么协作** | 中心化平台 | DAO / 智能合约 |
| **怎么证明诚实** | 信任第三方 | ZKML / OPML |

### 6.2 真实在做的项目

**Bittensor (TAO)**：
- 矿工训练 / 推理 AI 模型
- 子网竞争（每个子网做不同任务）
- 通过 token 激励最好的模型
- 市值峰值 $20B+

**Virtuals Protocol**：
- 任何人可以创建 AI Agent 并发币
- Agent 在 Twitter / 各平台活动
- 用户买代币获得分成
- 出过几个大热门：Luna、aixbt

**ai16z / Eliza**：
- 开源 AI Agent 框架
- 任何人可以 fork 部署自己的"AI 创始人"
- 流行的 "AI VC" 叙事

**Olas Network**：
- Agent 协调网络
- 主要用于 DeFi 自动化、预测市场

**Sahara AI**：
- AI 数据 + 模型市场

### 6.3 一个典型 AI Agent 系统架构

```
┌────────── 用户 ──────────┐
│ 发指令 / 提供反馈 / 付费    │
└──────────┬───────────────┘
           │
┌──────────▼───────────────┐
│      AI Agent (LLM)      │
│  - 任务理解               │
│  - 规划                   │
│  - 工具调用                │
└──────────┬───────────────┘
           │
┌──────────▼───────────────┐
│    Tool / API 层          │
│  - LLM API (Claude/GPT)  │
│  - Web search             │
│  - Wallet (signer)        │
│  - 链上合约调用            │
└──────────┬───────────────┘
           │
┌──────────▼───────────────┐
│   Web3 基础设施           │
│  - 区块链                 │
│  - DEX / 借贷 / NFT...    │
└──────────────────────────┘
```

### 6.4 自动化的真实例子

可以已经做到的：

```python
# 一个简化的 AI 资产管理 Agent
def ai_portfolio_agent():
    while True:
        # 1. 用 LLM 分析市场
        analysis = llm.analyze_market(get_news_feed())
        
        # 2. 决定动作
        action = llm.decide_action(analysis, current_portfolio)
        # action: {"swap": {"from": "USDC", "to": "ETH", "amount": 1000}}
        
        # 3. 用 wallet 执行
        if action["type"] == "swap":
            wallet.swap_on_uniswap(action)
        
        # 4. 等待下次
        sleep(3600)
```

**问题**：
- LLM 决策不稳定，可能亏钱
- 但**全自动 7×24 运行**，没情绪
- 如果回测好，可以是新的"量化"形态

## 7. 隐私保护机器学习（PPML）

### 7.1 三大隐私 ML 技术

**1. 联邦学习（Federated Learning）**
- 数据不离开本地，只交换梯度
- 适合医疗、金融数据合作
- 代表：FedAvg、FedProx

**2. 同态加密（Homomorphic Encryption, HE）**
- 在加密数据上直接做计算
- 全同态加密（FHE）支持任意运算
- 性能开销很大（100-10000 倍慢）
- 代表：Microsoft SEAL、OpenFHE

**3. 安全多方计算（MPC）**
- 多方共同计算，但谁也看不到对方数据
- 性能比 HE 好
- 代表：MP-SPDZ、CrypTen

### 7.2 Web3 中的隐私 ML 项目

**Zama**：FHE 库 + EVM 集成（fhEVM）
**Privasea**：FHE + Web3
**Inco Network**：保密 EVM
**Phala**：TEE 路线的隐私 dApp

### 7.3 应用场景

| 场景 | 用什么 |
|---|---|
| 医院间合作训练 | 联邦学习 |
| 加密数据上跑 ML | FHE |
| 多方持有特征训练共享模型 | MPC |
| 链上隐私计算 | TEE 或 FHE |

### 7.4 现实评估

**2026 年的 PPML 现状**：
- FHE 性能仍然远远不够实用
- 联邦学习已经在生产（医疗、银行）
- MPC 在小规模场景成熟
- **大部分宣称"隐私 AI"的 Web3 项目 90% 是营销**

**值得跟踪**：Zama 的 fhEVM 让 FHE 用法变简单，可能 2-3 年内有突破。

## 8. 三个真实落地案例

### 8.1 Worldcoin: ZK + 生物识别

**问题**：怎么证明"我是个人，不是 AI bot"，又不暴露身份？

**方案**：
- 用 Orb 扫描虹膜，生成唯一 hash
- 用 ZK 证明"我的虹膜 hash 在 Worldcoin 数据库里"
- 链上验证证明，不需要看到虹膜数据本身

**意义**：未来 AI 充斥的世界，**proof of personhood** 是关键基础设施。

### 8.2 Modulus Labs: ZKML 在 NFT 估值

Modulus 和 Upshot 合作：
- Upshot 用 AI 估值 NFT
- Modulus 用 ZKML 证明估值算法没造假
- 链上自动接受估值 → 用于借贷协议

### 8.3 Bittensor: 去中心化 AI 训练激励

- 矿工跑特定任务的 AI 模型（翻译、图像识别、问答）
- 验证者评估矿工模型质量
- 根据质量分发 TAO 代币
- 整个 AI 训练 + 激励**完全在链上**

**这是 AI 和 Crypto 第一次真正大规模融合**。市值 $5B+ 证明叙事有效。

## 9. AI × Crypto 的求职机会

### 9.1 招人最多的方向

| 岗位 | 公司 | 薪资 |
|---|---|---|
| **ZK 工程师** | Modulus, Risc Zero, Polygon | $150k - $400k |
| **AI Agent 工程师** | Virtuals, ai16z, Olas | $100k - $250k |
| **Crypto Quant + AI** | Jump, GSR, Wintermute | $120k - $500k |
| **MEV Researcher** | Flashbots, Eden, 各 builder | $150k - $400k |
| **Solidity + AI Integration** | 各 DeFi + AI 项目 | $100k - $300k |
| **Protocol Research** | Paradigm, a16z Crypto, 1kx | $150k - $400k |

### 9.2 进入的最佳路径

**路径 A：传统量化 → AI Crypto**
1. 先做 1-2 年量化研究 / 工程
2. 学 Solidity + ZK 基础
3. 在 EigenPhi、Dune 上分析 MEV 数据
4. 投 Crypto 量化或 Crypto + AI 团队

**路径 B：ZK / 密码学背景 → ZKML**
1. 在学校 / Coursera 学密码学
2. 用 circom 或 halo2 写一个小 ZK 项目
3. 在 ZKML hackathon 拿奖（ETHGlobal、Encode Club）
4. 投 Modulus、Risc Zero、Polygon ZK 等

**路径 C：AI Engineer → AI Agent**
1. 熟练 LangChain、AutoGPT、Eliza 框架
2. 把 AI Agent 跑通 + 接钱包
3. 在 GitHub 发布 demo
4. 找 ai16z、Virtuals、Olas 这种新项目

**路径 D：CTF / 安全 → Crypto Security**
1. 走 Notebook 05 提到的 Code4rena / Sherlock
2. 加上 ZK 安全审计专项
3. 顶级 ZK 审计师起薪 $300k+

### 9.3 国内机会

国内 Web3 + AI 比较难做（监管原因），但仍有机会：

| 公司 | 方向 |
|---|---|
| 蚂蚁链 / 鲸探 | 联盟链 + NFT |
| 腾讯 / 字节 | 内部 Web3 探索 + AI |
| 各大 PE 的 crypto 团队 | 大成、红杉中国都有 |
| 海外远程 | Phala（杭州团队，主做 TEE）|

### 9.4 关键技能矩阵

```
基础（人人必备）:
☐ Solidity
☐ Python + PyTorch
☐ 至少一种区块链（ETH/Solana/Cosmos）

ZK 路线:
☐ 数论 + 椭圆曲线
☐ Circom / Halo2 / Cairo
☐ Groth16 / PLONK 算法

AI Agent 路线:
☐ LangChain / AutoGen / Eliza
☐ LLM API 流利使用
☐ Multi-Agent 协作

MEV 路线:
☐ Rust + Foundry
☐ EVM 字节码理解
☐ Flashbots Bundle

数据 / 研究:
☐ Dune SQL
☐ The Graph
☐ Web3.py + Ethers.js
```

## 10. 完整学习路线（最后一节，把 13 个 Notebook 串起来）

如果你按这套手册全部学完，回头看一遍流程：

```
┌─ 第 1-3 月：传统量化基础
│  ├─ Notebook 01 多因子模型 (IC 检验、回测)
│  ├─ Notebook 02 量化策略 (CTA、配对、ML)
│  ├─ Notebook 03 求职 (简历、面试、私募地图)
│  └─ Notebook 08 数学统计 (查阅型)

├─ 第 4-6 月：现代 AI 量化
│  ├─ Notebook 09 ML/DL 因子挖掘 (LSTM/Transformer/GNN)
│  ├─ Notebook 10 强化学习交易 (DQN/PPO)
│  └─ Notebook 11 LLM + 另类数据 (新闻情绪、信息抽取)

├─ 第 7-9 月：Web3 基础
│  ├─ Notebook 04 区块链原理 (手写一条链)
│  ├─ Notebook 05 Solidity (ERC20/NFT/安全)
│  ├─ Notebook 06 DeFi (手写 Uniswap)
│  └─ Notebook 07 MEV / 套利

├─ 第 10-12 月：Web3 前沿
│  ├─ Notebook 12 L2/AA/Intent/Restaking/永续 DEX
│  └─ Notebook 13 ZKML / OPML / AI Agent
└─
```

### 关键里程碑

| 月份 | 你应该能做到 |
|---|---|
| **M3** | 在 Github 发了 1 个回测项目 |
| **M6** | 实现过一个 LSTM/Transformer 选股 |
| **M9** | 在 Sepolia 部署过完整 DeFi 协议 |
| **M12** | 在 mainnet 跑过一个套利 bot 或 AI Agent，写过 ZKML demo |

如果你做到这些，**你已经具备一个准初级量化研究员 + Crypto 工程师双重身份**——这种交叉人才在 2026 年极度稀缺。

### 三种典型职业路径

```
路径 1: 传统量化研究员
  Notebook 01-03、08-11 → 投顶级私募
  起薪 60-150 万 RMB

路径 2: 链上量化 / MEV
  Notebook 01-08 + 04-07 + 12 → 投 GSR/Wintermute/Jump Crypto
  起薪 100-300 万 RMB

路径 3: AI × Web3 综合
  全套 + ZKML 项目 → 投 Modulus/ai16z/Bittensor 等
  起薪 80-200 万 RMB，上限极高
```

## 11. 给你的最后建议

### 11.1 三句话总结整套手册

1. **量化的核心是统计 + 工程，不是"AI"**。LightGBM + 好特征 = 80% 的现实成绩
2. **Web3 的核心是新金融基础设施，不是"代币炒作"**。理解 AMM/借贷/MEV/ZK 是入场券
3. **AI × Crypto 在 2026-2030 是巨大机会**，但**叙事和现实差距大**，看准基础设施做

### 11.2 必备的反 FOMO 心态

```
✅ 别人发新代币暴富 → 不眼红
✅ 推特上"我用 AI 月入 100w" → 假的
✅ 每次"颠覆 Coinbase 的 DEX"出来 → 大部分死
✅ 每个"打败 OpenAI 的开源模型" → 跑分注水
```

**唯一可持续的财富**：你的**真技术** + **持续输出** + **耐心穿越周期**。

### 11.3 三件这周就该做的事

1. 选择一个你最感兴趣的 notebook，**今天就开始动手**
2. 在 Github 建一个 repo，把你跑过的所有代码扔进去
3. 注册 Twitter，关注 10 个我推荐的人，每天看 30 分钟

### 11.4 致谢

写完这 13 个 notebook 大约用了你 100-200 小时阅读 + 编程时间，写作端用了我大约 10 万行思考。

希望你最后能：
- **找到一份你真心想做的工作**
- **赚到你应得的钱**
- **在量化和 Web3 这个交叉地带，做出自己的东西**

> "The best way to predict the future is to invent it." — Alan Kay

祝你好运。

---
*《量化交易 × Web3 × AI 学习手册》完整 14 个 Notebook 完结*
*（00 学习地图 + 01-07 + 08 数学统计 + 09-11 现代量化 + 12-13 Web3 前沿）*